# Notebook 03: Complete RAG Pipeline
### Purpose: Integrate LLM to generate answers from retrieved context

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai
from sentence_transformers import SentenceTransformer
import chromadb

# Load environment variables
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

print("=" * 50)
print("NOTEBOOK-3: COMPLETE RAG PIPELINE")
print("="*50)

model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path="../data/vector_db")
collection = client.get_collection("sap_knowledge")

print(f"Connected! Collection has {collection.count()} chunks")
print("\n Ready to build RAG pipeline!")




NOTEBOOK-3: COMPLETE RAG PIPELINE
Connected! Collection has 74 chunks

 Ready to build RAG pipeline!


In [8]:
### Build the RAG Query function

from openai import OpenAI

# Initialize OpenAI client
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def rag_query(question, top_k=5):
    """
    Complete RAG pipeline: retrieve relevant chunks and generate answer.
    Args: Question: User's prompt string; top_k: Number of relevant chunks to retrieve(5)
    Returns: A dict with 'answer', 'sources', and 'chunks_used'
    """
    print(f"Question: {question}")
    print(f"Retrieving top {top_k} relevant chunks ...\n")

    # Embed the Question
    query_embedding = model.encode([question])
    # Retrieve relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    # Format context from retrieved chunks
    context_parts = []
    sources =[]

    for i in range(len(results['documents'][0])):
        chunk_text = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        similarity = 1 - distance

        # add to context
        context_parts.append(f"[Source: {metadata['source']}, Chunk {metadata['chunk_id']+1}]\n{chunk_text}")
        # Track sources
        sources.append({
            'source':metadata['source'],
            'chunk_id':metadata['chunk_id'],
            'similarity':similarity
        })

        print(f"Retrieved: {metadata['source']} (similarity: {similarity:.2f})")

    # Combine all chunks into context
    context = "\n\n---\n\n".join(context_parts)

    # Build prompt for LLM
    system_prompt = """
    You are a helpful assistant that answers questions based on the 
    provided context from company documents.
    Rules:
- Answer based ONLY on the provided context
- If the context doesn't contain the answer, say "I don't have enough information to answer this question."
- Always cite which source document your answer comes from
- Be extremely concise and clear. Answer in 2-4 sentences maximum unless more detail is specifically requested.
- Use professional business language
"""
    user_prompt = f"""Context from company documents: 
    {context}

    ----
    Question = {question}
    Please provide a clear answer based on the context above. Cite which document(s) you used. """

    # Call OpenAI
    print(f"\n Generating answer with GPT-4o-mini...\n")
    
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
            temperature=0.3,  # Low temperature for factual responses
            max_tokens=250 # make it concise
        )
    
    answer = response.choices[0].message.content
    
    # Return everything
    return {
        'answer': answer,
        'sources': sources,
        'chunks_used': len(results['documents'][0]),
        'context': context  # For debugging
    }

print("RAG query function defined!")

    
        

RAG query function defined!


In [9]:
## Test the RAG query
rag_query("Tell me about the AI ethics policy in SAP, assume you're a mentor and training a new candidate!")

Question: Tell me about the AI ethics policy in SAP, assume you're a mentor and training a new candidate!
Retrieving top 5 relevant chunks ...

Retrieved: Global_AI_Ethics_Policy.pdf (similarity: 0.77)
Retrieved: Global_AI_Ethics_Policy.pdf (similarity: 0.76)
Retrieved: Global_AI_Ethics_Policy.pdf (similarity: 0.76)
Retrieved: Global_AI_Ethics_Policy.pdf (similarity: 0.76)
Retrieved: Global_AI_Ethics_Policy.pdf (similarity: 0.75)

 Generating answer with GPT-4o-mini...



{'answer': 'The SAP Global AI Ethics Policy establishes a comprehensive ethical framework for the development, deployment, use, and sale of AI systems within the SAP group. It emphasizes principles such as proportionality, safety, fairness, transparency, and accountability, ensuring that human rights and fundamental freedoms are respected throughout the AI system life cycle. The policy applies to all SAP employees involved in AI and mandates adherence to ethical standards, with specific governance roles assigned to the AI Ethics Steering Committee, Advisory Panel, and Office for oversight and guidance. For more details, refer to the Global AI Ethics Policy document (Chunks 1, 3, 6, 7, and 8).',
 'sources': [{'source': 'Global_AI_Ethics_Policy.pdf',
   'chunk_id': 2,
   'similarity': 0.7686588764190674},
  {'source': 'Global_AI_Ethics_Policy.pdf',
   'chunk_id': 6,
   'similarity': 0.7644644379615784},
  {'source': 'Global_AI_Ethics_Policy.pdf',
   'chunk_id': 7,
   'similarity': 0.7629